In [ ]:
# NoBenGraham-ResNet-18: ResNet-18, plain BCE, balanced dataset, normalized (no Ben Graham) images.
# Prints: the device, the run name and the recipe summary.
# Expected: "Device: cuda" (set Settings > Accelerator > GPU T4 x2 if it says cpu).
import os
import time
import json
import random
import zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

# ---------------- SETTINGS (change things here only) ----------------
DATASET_NAME = "balanced"
ZIP_PATH = "/kaggle/input/notebooks/lohitashwamadhan/02-preprocessing/_output_.zip"
EXTRACT_DIR = "/kaggle/temp/data"
ROOT = EXTRACT_DIR
IMAGE_FOLDER = "processed"            # "processed" = normalized, NO Ben Graham
RUN_NAME = "resnet18_plainbce_balanced_normalized_512"
MODEL_NAME = "ResNet-18"
PREPROCESSING = "Normalized"
LOSS_NAME = "plain_bce"

SEED = 42
IMAGE_SIZE = 512
BATCH_SIZE = 32
NUM_WORKERS = 4
USE_AUGMENTATION = False
USE_AMP = True                        # mixed precision, faster on the T4. Only used on a GPU.

# Stopping rule: early stopping on val AUROC. MAX_EPOCHS is only a safety cap.
MAX_EPOCHS = 30
PATIENCE = 5

# Fine-tuning recipe (copied from Kabir's RETFound recipe, to confirm with him)
HEAD_ONLY_EPOCHS = 1                  # epoch 1: only the new fc layer learns
BASE_LR = 6e-4                        # learning rate of the top group (fc)
LAYER_DECAY = 0.65                    # each group further from fc gets 0.65x the lr of the one above
WEIGHT_DECAY = 0.05                   # confirmed: weight decay 0.05
# Optimizer is AdamW (confirmed).

# Where the ImageNet weights come from:
#   "download" = torchvision downloads them (needs Internet ON in notebook settings)
#   "input"    = load from WEIGHTS_FILE (attach a ResNet-18 weights file as an input)
#   "none"     = random weights (only for the local smoke test)
PRETRAINED_SOURCE = "download"
WEIGHTS_FILE = "/kaggle/input/resnet18-weights/resnet18-f37072fd.pth"

TARGET_SENSITIVITY = 0.90
BOOTSTRAP_N = 1000
SAMPLE_FRACTION = 0.10                # sample run uses 10% of train and val
SAMPLE_EPOCHS = 3
RESUME = False                        # True = continue from last_<RUN>.pt after a crash

OUT_DIR = "/kaggle/working"
FIG_DIR = OUT_DIR + "/figures"
# --------------------------------------------------------------------

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
COLORS = {"blue": "#2E5EAA", "orange": "#E4572E", "teal": "#17A398",
          "amber": "#F2A541", "purple": "#7A4EAB", "gray": "#6C757D"}

plt.rcParams.update({
    "font.size": 12, "axes.titlesize": 13, "axes.labelsize": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "savefig.dpi": 200,
    "figure.dpi": 100,
})

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def add_footer(fig, run):
    text = "Run: " + run + " | Dataset: " + DATASET_NAME + " | Folder: " + IMAGE_FOLDER + " | Seed: " + str(SEED)
    fig.text(0.01, 0.005, text, fontsize=8, color=COLORS["gray"])


def save_fig(fig, number, name, run):
    add_footer(fig, run)
    fig.tight_layout(rect=[0, 0.03, 1, 1])
    path = FIG_DIR + "/" + number + "_" + name + "_" + run + ".png"
    fig.savefig(path)
    plt.show()
    print("Saved", path)


set_seed(SEED)
print("Device:", device)
print("Run:", RUN_NAME)
print("Recipe: epoch 1 head only, then all layers with layer-wise lr decay", LAYER_DECAY,
      "from lr", BASE_LR, "(cosine). Early stopping patience", PATIENCE, "cap", MAX_EPOCHS)


In [ ]:
# Prints: extraction progress, then split counts, shared group count and missing file count.
# Expected: train 30287, val 6483, test 6491, shared groups 0 / 0 / 0, missing files 0.
# Takes a few minutes the first time. /kaggle/temp is wiped on restart, so rerun after a restart.
os.makedirs(EXTRACT_DIR, exist_ok=True)

zf = zipfile.ZipFile(ZIP_PATH, "r")
names = zf.namelist()
prefix_train = IMAGE_FOLDER + "/train/"
prefix_val = IMAGE_FOLDER + "/val/"
prefix_test = IMAGE_FOLDER + "/test/"

wanted_names = []
for name in names:
    if name == "manifest_v2.csv":
        wanted_names.append(name)
    elif name.startswith(prefix_train) or name.startswith(prefix_val) or name.startswith(prefix_test):
        if not name.endswith("/"):
            wanted_names.append(name)

count = 0
for name in wanted_names:
    target = EXTRACT_DIR + "/" + name
    if not os.path.exists(target):
        zf.extract(name, EXTRACT_DIR)
    count = count + 1
    if count % 5000 == 0:
        print("Done", count, "of", len(wanted_names))
zf.close()
print("Extraction finished:", len(wanted_names), "files wanted")

manifest = pd.read_csv(EXTRACT_DIR + "/manifest_v2.csv")
print("Manifest rows:", len(manifest))
ok_mask = manifest["load_ok"].astype(str).str.lower() == "true"
split_mask = manifest["split"].isin(["train", "val", "test"])
manifest = manifest[ok_mask & split_mask].reset_index(drop=True)

train_df = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_df = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_df = manifest[manifest["split"] == "test"].reset_index(drop=True)

expected = {"train": 30287, "val": 6483, "test": 6491}
all_good = True
for split_name, frame in [("train", train_df), ("val", val_df), ("test", test_df)]:
    folder = ROOT + "/" + IMAGE_FOLDER + "/" + split_name
    if not os.path.isdir(folder):
        print("FOLDER MISSING:", folder)
        all_good = False
        continue
    missing = 0
    for fname in frame["filename"].tolist():
        if not os.path.exists(folder + "/" + fname):
            missing = missing + 1
    n1 = int((frame["label"] == 1).sum())
    n0 = int((frame["label"] == 0).sum())
    print(split_name, "images:", len(frame), "| label 0:", n0, "| label 1:", n1,
          "| expected", expected[split_name], "| missing files:", missing)
    if len(frame) != expected[split_name] or missing > 0:
        all_good = False

g_train = set(train_df["group_id"].tolist())
g_val = set(val_df["group_id"].tolist())
g_test = set(test_df["group_id"].tolist())
shared_tv = len(g_train & g_val)
shared_tt = len(g_train & g_test)
shared_vt = len(g_val & g_test)
print("Shared groups train/val:", shared_tv, "| train/test:", shared_tt, "| val/test:", shared_vt)
if shared_tv + shared_tt + shared_vt > 0:
    all_good = False

if all_good:
    print("DATA CHECK PASSED")
else:
    print("DATA CHECK FAILED - stop and report this to the team before training")


In [ ]:
# Prints: one batch shape and the label counts in it.
# Expected: images torch.Size([32, 3, 224, 224]), labels torch.Size([32]).
class RetinaDataset(Dataset):
    def __init__(self, df, image_folder, train=False):
        self.filenames = df["filename"].tolist()
        self.labels = df["label"].astype(int).tolist()
        self.sources = df["source"].tolist()
        self.groups = df["group_id"].tolist()
        self.grades = df["grade"].tolist()
        splits = df["split"].tolist()
        self.paths = []
        for i in range(len(self.filenames)):
            self.paths.append(ROOT + "/" + image_folder + "/" + splits[i] + "/" + self.filenames[i])

        steps = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]
        if train and USE_AUGMENTATION:
            steps.append(transforms.RandomHorizontalFlip())
        steps.append(transforms.ToTensor())
        steps.append(transforms.Normalize(MEAN, STD))
        self.transform = transforms.Compose(steps)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        image = Image.open(self.paths[index]).convert("RGB")
        image = self.transform(image)
        return image, self.labels[index]


def make_loader(df, shuffle, train):
    dataset = RetinaDataset(df, IMAGE_FOLDER, train=train)
    generator = torch.Generator()
    generator.manual_seed(SEED)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                        num_workers=NUM_WORKERS, pin_memory=(device.type == "cuda"),
                        drop_last=train, generator=generator)
    return loader


check_loader = make_loader(train_df.head(BATCH_SIZE * 2), shuffle=False, train=True)
check_images, check_labels = next(iter(check_loader))
print("images", check_images.shape, "| labels", check_labels.shape)


In [ ]:
# Prints: parameter count and which groups are trainable in each phase.
# Expected: about 11.2 million parameters in total; phase 1 trains 513 (the fc layer),
# phase 2 trains all of them. If this cell fails with a download error, Internet is off:
# turn it on in notebook settings, or attach weights and set PRETRAINED_SOURCE = "input".
def build_model():
    if PRETRAINED_SOURCE == "download":
        try:
            model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        except Exception as error:
            print("COULD NOT DOWNLOAD IMAGENET WEIGHTS. Internet is probably OFF.")
            print("Fix: turn Internet on in the notebook settings, or attach a ResNet-18 weights")
            print("file as an input and set PRETRAINED_SOURCE = 'input' and WEIGHTS_FILE in Cell 1.")
            raise error
    elif PRETRAINED_SOURCE == "input":
        model = models.resnet18(weights=None)
        state = torch.load(WEIGHTS_FILE, map_location="cpu")
        model.load_state_dict(state)
    else:
        model = models.resnet18(weights=None)
    model.fc = nn.Linear(512, 1)
    return model


def get_layer_groups(model):
    # Six groups in order, from the input side to the output side.
    groups = []
    groups.append(list(model.conv1.parameters()) + list(model.bn1.parameters()))
    groups.append(list(model.layer1.parameters()))
    groups.append(list(model.layer2.parameters()))
    groups.append(list(model.layer3.parameters()))
    groups.append(list(model.layer4.parameters()))
    groups.append(list(model.fc.parameters()))
    return groups


def set_train_mode(model, head_only):
    model.train()
    if head_only:
        # Frozen layers stay in eval mode so their BatchNorm statistics do not change.
        for name, module in model.named_children():
            if name != "fc":
                module.eval()


def make_optimizer(model, epoch, max_epochs):
    # Returns (optimizer, scheduler). Also sets which parameters are frozen for this phase.
    # The optimizer is rebuilt when the phase changes so newly unfrozen layers get updated.
    if epoch <= HEAD_ONLY_EPOCHS:
        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():
            param.requires_grad = True
        optimizer = torch.optim.AdamW(model.fc.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
        return optimizer, None

    for param in model.parameters():
        param.requires_grad = True
    groups = get_layer_groups(model)
    last_index = len(groups) - 1
    param_groups = []
    for i in range(len(groups)):
        lr_group = BASE_LR * (LAYER_DECAY ** (last_index - i))
        param_groups.append({"params": groups[i], "lr": lr_group})
    optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
    t_max = max(1, max_epochs - HEAD_ONLY_EPOCHS)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max)
    return optimizer, scheduler


model_check = build_model()
total_params = 0
for p in model_check.parameters():
    total_params = total_params + p.numel()
print("Total parameters:", total_params)
print("Phase 1 (epoch 1 to", HEAD_ONLY_EPOCHS, "): only fc trains, 513 parameters")
group_names = ["conv1+bn1", "layer1", "layer2", "layer3", "layer4", "fc"]
for i in range(6):
    print("Phase 2 lr for", group_names[i], "=", BASE_LR * (LAYER_DECAY ** (5 - i)))
del model_check


In [ ]:
# Prints nothing. Defines get_predictions, count_outcomes, compute_rates, pick_threshold, bootstrap_ci.
def get_predictions(model, loader):
    # Returns probabilities, true labels and the average loss (on the true 0/1 labels).
    model.eval()
    criterion = nn.BCEWithLogitsLoss(reduction="sum")
    all_probs = []
    all_labels = []
    total_loss = 0.0
    use_amp = USE_AMP and device.type == "cuda"
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            labels_f = labels.float().unsqueeze(1).to(device)
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(images)
            logits = logits.float()
            total_loss = total_loss + criterion(logits, labels_f).item()
            probs = torch.sigmoid(logits).squeeze(1).cpu().numpy()
            for value in probs:
                all_probs.append(float(value))
            for value in labels.numpy():
                all_labels.append(int(value))
    avg_loss = total_loss / max(1, len(all_labels))
    return np.array(all_probs), np.array(all_labels), avg_loss


def count_outcomes(labels, predicted):
    tp = int(((labels == 1) & (predicted == 1)).sum())
    tn = int(((labels == 0) & (predicted == 0)).sum())
    fp = int(((labels == 0) & (predicted == 1)).sum())
    fn = int(((labels == 1) & (predicted == 0)).sum())
    return tn, fp, fn, tp


def compute_rates(tn, fp, fn, tp):
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0.0
    return sens, spec, prec, f1


def safe_auroc(labels, probs):
    if len(np.unique(labels)) < 2:
        return 0.5
    return float(roc_auc_score(labels, probs))


def pick_threshold(labels, probs, target):
    # Scan cutoffs 0.000 to 1.000 in steps of 0.001. Keep the HIGHEST cutoff with sensitivity >= target.
    positives = probs[labels == 1]
    negatives = probs[labels == 0]
    best_cut = 0.0
    found = False
    for step in range(0, 1001):
        cut = step / 1000.0
        sens = float((positives >= cut).mean())
        if sens >= target:
            best_cut = cut
            found = True
    if not found:
        print("WARNING: no cutoff reached the target sensitivity, using 0.0")
    sens = float((positives >= best_cut).mean())
    spec = float((negatives < best_cut).mean())
    return best_cut, sens, spec


def bootstrap_ci(labels, probs, groups, cutoff, n_resamples, seed):
    # Resample whole patient groups with replacement. 95 percent percentile intervals.
    rng = np.random.RandomState(seed)
    group_to_rows = {}
    for i in range(len(groups)):
        key = groups[i]
        if key not in group_to_rows:
            group_to_rows[key] = []
        group_to_rows[key].append(i)
    group_keys = list(group_to_rows.keys())
    group_arrays = []
    for key in group_keys:
        group_arrays.append(np.array(group_to_rows[key]))

    names = ["sensitivity", "specificity", "precision", "f1", "auroc", "pr_auc"]
    collected = {}
    for name in names:
        collected[name] = []

    for r in range(n_resamples):
        picks = rng.randint(0, len(group_keys), size=len(group_keys))
        pieces = []
        for p in picks:
            pieces.append(group_arrays[p])
        rows = np.concatenate(pieces)
        l = labels[rows]
        pr = probs[rows]
        if len(np.unique(l)) < 2:
            continue
        predicted = (pr >= cutoff).astype(int)
        tn, fp, fn, tp = count_outcomes(l, predicted)
        sens, spec, prec, f1 = compute_rates(tn, fp, fn, tp)
        collected["sensitivity"].append(sens)
        collected["specificity"].append(spec)
        collected["precision"].append(prec)
        collected["f1"].append(f1)
        collected["auroc"].append(float(roc_auc_score(l, pr)))
        collected["pr_auc"].append(float(average_precision_score(l, pr)))

    result = {}
    for name in names:
        values = np.array(collected[name])
        result[name] = (float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5)))
    return result


print("Helper functions defined")


In [ ]:
# Prints: one line per epoch (loss, val loss, val AUROC, sens/spec at 0.5, lr, seconds) and a RESULT line.
# Defines train_model(sample_run). Does not train by itself.
def train_model(sample_run):
    set_seed(SEED)
    run = RUN_NAME
    max_epochs = MAX_EPOCHS
    tr_df = train_df
    va_df = val_df
    if sample_run:
        run = RUN_NAME + "_sample"
        max_epochs = SAMPLE_EPOCHS
        tr_df = train_df.sample(frac=SAMPLE_FRACTION, random_state=SEED).reset_index(drop=True)
        va_df = val_df.sample(frac=SAMPLE_FRACTION, random_state=SEED).reset_index(drop=True)
        print("SAMPLE RUN:", len(tr_df), "train images,", len(va_df), "val images,", max_epochs, "epochs")

    train_loader = make_loader(tr_df, shuffle=True, train=True)
    val_loader = make_loader(va_df, shuffle=False, train=False)

    model = build_model().to(device)
    criterion = nn.BCEWithLogitsLoss()
    use_amp = USE_AMP and device.type == "cuda"
    scaler = torch.amp.GradScaler(device.type, enabled=use_amp)

    best_path = OUT_DIR + "/best_" + run + ".pt"
    last_path = OUT_DIR + "/last_" + run + ".pt"
    history_path = OUT_DIR + "/history_" + run + ".csv"

    history = []
    best_auc = -1.0
    best_epoch = 0
    bad_epochs = 0
    start_epoch = 1
    stop_reason = "hit the safety cap"

    optimizer, scheduler = make_optimizer(model, 1, max_epochs)

    if RESUME and os.path.exists(last_path):
        saved = torch.load(last_path, map_location=device)
        model.load_state_dict(saved["model"])
        history = saved["history"]
        best_auc = saved["best_auc"]
        best_epoch = saved["best_epoch"]
        bad_epochs = saved["bad_epochs"]
        start_epoch = saved["epoch"] + 1
        optimizer, scheduler = make_optimizer(model, start_epoch, max_epochs)
        optimizer.load_state_dict(saved["optimizer"])
        if scheduler is not None and saved["scheduler"] is not None:
            scheduler.load_state_dict(saved["scheduler"])
        print("RESUMED from epoch", saved["epoch"])

    for epoch in range(start_epoch, max_epochs + 1):
        start_time = time.time()

        if epoch == HEAD_ONLY_EPOCHS + 1:
            optimizer, scheduler = make_optimizer(model, epoch, max_epochs)
        head_only = epoch <= HEAD_ONLY_EPOCHS
        set_train_mode(model, head_only)

        total_loss = 0.0
        seen = 0
        for images, labels in train_loader:
            images = images.to(device, non_blocking=True)
            labels_f = labels.float().unsqueeze(1).to(device)
            optimizer.zero_grad()
            with torch.autocast(device_type=device.type, enabled=use_amp):
                logits = model(images)
                loss = criterion(logits.float(), labels_f)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss = total_loss + loss.item() * images.size(0)
            seen = seen + images.size(0)
        train_loss = total_loss / max(1, seen)

        current_lr = optimizer.param_groups[len(optimizer.param_groups) - 1]["lr"]
        if scheduler is not None:
            scheduler.step()

        probs, labels_arr, val_loss = get_predictions(model, val_loader)
        val_auc = safe_auroc(labels_arr, probs)
        predicted = (probs >= 0.5).astype(int)
        tn, fp, fn, tp = count_outcomes(labels_arr, predicted)
        sens, spec, prec, f1 = compute_rates(tn, fp, fn, tp)
        seconds = time.time() - start_time

        row = {"epoch": epoch, "phase": "head_only" if head_only else "all_layers",
               "train_loss": train_loss, "val_loss": val_loss, "val_auroc": val_auc,
               "val_sens_at_0.5": sens, "val_spec_at_0.5": spec,
               "lr_fc": current_lr, "sec_per_epoch": seconds}
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)

        marker = ""
        if val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch
            bad_epochs = 0
            torch.save(model.state_dict(), best_path)
            marker = "  <- new best"
        else:
            bad_epochs = bad_epochs + 1

        print("epoch", epoch, "|", row["phase"], "| train loss", round(train_loss, 4),
              "| val loss", round(val_loss, 4), "| val AUROC", round(val_auc, 4),
              "| sens", round(sens, 3), "| spec", round(spec, 3),
              "| lr_fc", format(current_lr, ".2e"), "|", round(seconds), "s" + marker)

        torch.save({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict() if scheduler is not None else None,
                    "history": history, "best_auc": best_auc, "best_epoch": best_epoch,
                    "bad_epochs": bad_epochs, "epoch": epoch}, last_path)

        if bad_epochs >= PATIENCE:
            stop_reason = "early stopping (patience " + str(PATIENCE) + ")"
            print("Early stop: no better val AUROC for", PATIENCE, "epochs")
            break

    print("RESULT", run, "| seed", SEED, "| best epoch", best_epoch, "| val AUROC", round(best_auc, 4),
          "| stopped by:", stop_reason)
    return run, stop_reason


In [ ]:
# Prints: a few epochs on 10% of the data and a RESULT line. Run this BEFORE the full run.
# Expected: no error, val AUROC above 0.5 at the end. "sens 1.0, spec 0.0" in an early epoch is normal.
LAST_RUN, LAST_STOP = train_model(True)


In [ ]:
# Prints: one line per epoch and a RESULT line. Long cell (hours). Do NOT use Run All.
# Expected: stops by patience, not by the cap. If it says "hit the safety cap", tell Mohit.
# If the session crashes: set RESUME = True in Cell 1, rerun Cells 1 to 6, then run this cell again.
LAST_RUN, LAST_STOP = train_model(False)


In [ ]:
# Prints: threshold, val sensitivity and specificity at that threshold, val AUROC.
# Expected: val sensitivity just above 0.90, threshold saved to threshold_<RUN>.json.
SAMPLE_MODE = LAST_RUN.endswith("_sample")
history_df = pd.read_csv(OUT_DIR + "/history_" + LAST_RUN + ".csv")

best_model = build_model().to(device)
best_model.load_state_dict(torch.load(OUT_DIR + "/best_" + LAST_RUN + ".pt", map_location=device))

val_loader_full = make_loader(val_df, shuffle=False, train=False)
val_probs, val_labels, val_loss_full = get_predictions(best_model, val_loader_full)
val_auc_final = safe_auroc(val_labels, val_probs)

threshold, val_sens, val_spec = pick_threshold(val_labels, val_probs, TARGET_SENSITIVITY)
with open(OUT_DIR + "/threshold_" + LAST_RUN + ".json", "w") as f:
    json.dump({"threshold": threshold, "val_sensitivity": val_sens,
               "val_specificity": val_spec, "target": TARGET_SENSITIVITY}, f)

print("Val AUROC:", round(val_auc_final, 4))
print("Threshold (highest cutoff with val sensitivity >= target):", threshold)
print("Val sensitivity:", round(val_sens, 4), "| Val specificity:", round(val_spec, 4))
print("This threshold is now frozen.")


In [ ]:
# Prints: test metrics, confusion matrix, 95% CIs, breakdown by source and grade, and the TABLE ROW.
# Expected: test sensitivity near 0.88 to 0.90, then a table row in the team's column layout.
with open(OUT_DIR + "/threshold_" + LAST_RUN + ".json", "r") as f:
    frozen = json.load(f)
threshold = frozen["threshold"]

test_loader_full = make_loader(test_df, shuffle=False, train=False)
test_probs, test_labels, test_loss = get_predictions(best_model, test_loader_full)
test_pred = (test_probs >= threshold).astype(int)

tn, fp, fn, tp = count_outcomes(test_labels, test_pred)
t_sens, t_spec, t_prec, t_f1 = compute_rates(tn, fp, fn, tp)
t_auc = safe_auroc(test_labels, test_probs)
t_prauc = float(average_precision_score(test_labels, test_probs))

print("TEST RESULTS at threshold", threshold)
print("Sensitivity:", round(t_sens, 4), "| Specificity:", round(t_spec, 4))
print("Precision:", round(t_prec, 4), "| F1:", round(t_f1, 4))
print("AUROC:", round(t_auc, 4), "| PR-AUC:", round(t_prauc, 4))
print("Confusion matrix -> TN:", tn, "| FP:", fp, "| FN:", fn, "| TP:", tp)

print("Bootstrap,", BOOTSTRAP_N, "resamples by patient group ...")
test_groups = test_df["group_id"].tolist()
ci = bootstrap_ci(test_labels, test_probs, test_groups, threshold, BOOTSTRAP_N, SEED)
for name in ["sensitivity", "specificity", "precision", "f1", "auroc", "pr_auc"]:
    print(name, "95% CI:", round(ci[name][0], 4), "to", round(ci[name][1], 4))

# Breakdown by source
test_sources = np.array(test_df["source"].tolist())
source_rows = []
for src in sorted(set(test_sources.tolist())):
    mask = test_sources == src
    s_labels = test_labels[mask]
    s_pred = test_pred[mask]
    s_probs = test_probs[mask]
    a, b, c, d = count_outcomes(s_labels, s_pred)
    s_sens, s_spec, s_prec, s_f1 = compute_rates(a, b, c, d)
    source_rows.append({"source": src, "n": int(mask.sum()), "sensitivity": s_sens,
                        "specificity": s_spec, "auroc": safe_auroc(s_labels, s_probs)})
    print("Source", src, "(n", int(mask.sum()), ") sens", round(s_sens, 3), "spec", round(s_spec, 3),
          "AUROC", round(safe_auroc(s_labels, s_probs), 3))

# Sensitivity by grade among label 1 images
test_grades = np.array(test_df["grade"].tolist(), dtype=float)
grade_rows = []
for g in sorted(set(test_grades[test_labels == 1].tolist())):
    if np.isnan(g):
        continue
    mask = (test_grades == g) & (test_labels == 1)
    caught = int((test_pred[mask] == 1).sum())
    total = int(mask.sum())
    grade_rows.append({"grade": int(g), "caught": caught, "total": total,
                       "sensitivity": caught / total if total > 0 else 0.0})
    print("Grade", int(g), ":", caught, "of", total, "=", round(caught / max(1, total), 3))

# Save predictions
pred_df = pd.DataFrame({"filename": test_df["filename"].tolist(), "source": test_df["source"].tolist(),
                        "group_id": test_df["group_id"].tolist(), "grade": test_df["grade"].tolist(),
                        "label": test_labels.tolist(), "probability": test_probs.tolist(),
                        "predicted": test_pred.tolist()})
pred_df.to_csv(OUT_DIR + "/test_predictions_" + LAST_RUN + ".csv", index=False)

# Results row
best_idx = int(history_df["val_auroc"].idxmax())
best_epoch_final = int(history_df.loc[best_idx, "epoch"])
results_row = {
    "run": LAST_RUN, "model": MODEL_NAME, "preprocessing": PREPROCESSING,
    "dataset_name": DATASET_NAME, "image_folder": IMAGE_FOLDER, "loss": LOSS_NAME, "seed": SEED,
    "test_auroc": t_auc, "test_auroc_ci_low": ci["auroc"][0], "test_auroc_ci_high": ci["auroc"][1],
    "test_sensitivity": t_sens, "test_specificity": t_spec,
    "test_precision": t_prec, "test_f1": t_f1, "test_pr_auc": t_prauc,
    "val_auroc": float(history_df.loc[best_idx, "val_auroc"]),
    "val_sensitivity": val_sens, "val_specificity": val_spec, "threshold": threshold,
    "best_epoch": best_epoch_final, "epochs_run": int(len(history_df)),
    "sec_per_epoch": float(history_df["sec_per_epoch"].mean()),
    "stopped_by": LAST_STOP, "tn": tn, "fp": fp, "fn": fn, "tp": tp,
}
pd.DataFrame([results_row]).to_csv(OUT_DIR + "/results_row_" + LAST_RUN + ".csv", index=False)

# Table row in the same columns as the team's Google Doc table
table_row = {
    "Model": MODEL_NAME, "Preprocessing": PREPROCESSING,
    "Test AUROC": round(t_auc, 3), "Sensitivity": round(t_sens, 3),
    "Specificity @ 90% sens": round(t_spec, 3), "Val AUROC": round(results_row["val_auroc"], 3),
    "PR-AUC": round(t_prauc, 3), "Precision": round(t_prec, 3), "F1": round(t_f1, 3),
    "Best epoch": best_epoch_final, "Sec / epoch": int(round(results_row["sec_per_epoch"])),
    "Notes": "plain BCE, balanced, 512px, " + LAST_STOP,
}
table_df = pd.DataFrame([table_row])
table_df.to_csv(OUT_DIR + "/team_table_row_" + LAST_RUN + ".csv", index=False)
print("")
print("TEAM TABLE ROW (copy into the Google Doc):")
print(table_df.to_string(index=False))

print("")
print("TABLE ROW:", MODEL_NAME, "/", PREPROCESSING, "| test AUROC", round(t_auc, 3),
      "| sens", round(t_sens, 3), "| spec @ 90% sens", round(t_spec, 3),
      "| val AUROC", round(results_row["val_auroc"], 3), "| PR-AUC", round(t_prauc, 3),
      "| precision", round(t_prec, 3), "| F1", round(t_f1, 3),
      "| best epoch", best_epoch_final, "| sec/epoch", round(results_row["sec_per_epoch"]))


In [ ]:
# Prints: "Saved <path>" for each of the 7 figures, and shows them.
# Expected: files 01 to 07 in /kaggle/working/figures.
run = LAST_RUN
op_color = COLORS["orange"]

# 01 training curves
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
axes[0].plot(history_df["epoch"], history_df["train_loss"], color=COLORS["blue"], label="train loss")
axes[0].plot(history_df["epoch"], history_df["val_loss"], color=COLORS["orange"], label="val loss")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[1].plot(history_df["epoch"], history_df["val_auroc"], color=COLORS["teal"], marker="o")
axes[1].axvline(best_epoch_final, color=COLORS["amber"], linestyle="--", label="best epoch " + str(best_epoch_final))
axes[1].set_title("Validation AUROC")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[2].plot(history_df["epoch"], history_df["val_sens_at_0.5"], color=COLORS["purple"], label="sensitivity")
axes[2].plot(history_df["epoch"], history_df["val_spec_at_0.5"], color=COLORS["gray"], label="specificity")
axes[2].set_title("Val sens / spec at cutoff 0.5 (monitoring only)")
axes[2].set_xlabel("Epoch")
axes[2].legend()
save_fig(fig, "01", "training_curves", run)

# 02 validation
fpr, tpr, _ = roc_curve(val_labels, val_probs)
fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))
axes[0].plot(fpr, tpr, color=COLORS["blue"], label="AUROC " + str(round(val_auc_final, 3)))
axes[0].plot([0, 1], [0, 1], color=COLORS["gray"], linestyle=":")
axes[0].scatter([1 - val_spec], [val_sens], color=op_color, zorder=5, label="chosen cutoff " + str(threshold))
axes[0].set_title("Validation ROC")
axes[0].set_xlabel("1 - specificity")
axes[0].set_ylabel("Sensitivity")
axes[0].legend(loc="lower right")
cuts = np.linspace(0, 1, 1001)
sens_curve = []
spec_curve = []
for cut in cuts:
    sens_curve.append(float((val_probs[val_labels == 1] >= cut).mean()))
    spec_curve.append(float((val_probs[val_labels == 0] < cut).mean()))
axes[1].plot(cuts, sens_curve, color=COLORS["teal"], label="sensitivity")
axes[1].plot(cuts, spec_curve, color=COLORS["purple"], label="specificity")
axes[1].axhline(TARGET_SENSITIVITY, color=op_color, linestyle="--", label="target 0.90")
axes[1].axvline(threshold, color=COLORS["gray"], linestyle=":")
axes[1].set_title("Sensitivity and specificity vs cutoff")
axes[1].set_xlabel("Cutoff")
axes[1].legend()
axes[2].hist(val_probs[val_labels == 0], bins=40, alpha=0.6, color=COLORS["blue"], label="healthy")
axes[2].hist(val_probs[val_labels == 1], bins=40, alpha=0.6, color=COLORS["orange"], label="referable DR")
axes[2].axvline(threshold, color=COLORS["gray"], linestyle="--")
axes[2].set_title("Validation predicted probability")
axes[2].set_xlabel("Probability")
axes[2].legend()
save_fig(fig, "02", "validation", run)

# 03 test ROC and precision-recall
fpr_t, tpr_t, _ = roc_curve(test_labels, test_probs)
prec_c, rec_c, _ = precision_recall_curve(test_labels, test_probs)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(fpr_t, tpr_t, color=COLORS["blue"], label="AUROC " + str(round(t_auc, 3)))
axes[0].plot([0, 1], [0, 1], color=COLORS["gray"], linestyle=":")
axes[0].scatter([1 - t_spec], [t_sens], color=op_color, zorder=5, label="operating point")
axes[0].set_title("Test ROC")
axes[0].set_xlabel("1 - specificity")
axes[0].set_ylabel("Sensitivity")
axes[0].legend(loc="lower right")
axes[1].plot(rec_c, prec_c, color=COLORS["teal"], label="PR-AUC " + str(round(t_prauc, 3)))
axes[1].scatter([t_sens], [t_prec], color=op_color, zorder=5, label="operating point")
axes[1].set_title("Test precision-recall")
axes[1].set_xlabel("Recall (sensitivity)")
axes[1].set_ylabel("Precision")
axes[1].legend(loc="upper right")
save_fig(fig, "03", "test_roc_pr", run)

# 04 test confusion matrix and probability histogram
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
matrix = np.array([[tn, fp], [fn, tp]])
axes[0].imshow(matrix, cmap="Blues")
axes[0].grid(False)
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, str(matrix[i, j]), ha="center", va="center", fontsize=16,
                     color="white" if matrix[i, j] > matrix.max() / 2 else "black")
axes[0].set_xticks([0, 1])
axes[0].set_yticks([0, 1])
axes[0].set_xticklabels(["Predicted healthy", "Predicted DR"])
axes[0].set_yticklabels(["Actual healthy", "Actual DR"])
axes[0].set_title("Test confusion matrix")
axes[1].hist(test_probs[test_labels == 0], bins=40, alpha=0.6, color=COLORS["blue"], label="healthy")
axes[1].hist(test_probs[test_labels == 1], bins=40, alpha=0.6, color=COLORS["orange"], label="referable DR")
axes[1].axvline(threshold, color=COLORS["gray"], linestyle="--")
axes[1].set_title("Test predicted probability")
axes[1].set_xlabel("Probability")
axes[1].legend()
save_fig(fig, "04", "test_confusion_hist", run)

# 05 performance by source and sensitivity by grade
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
src_names = []
src_sens = []
src_spec = []
for r in source_rows:
    src_names.append(r["source"] + " (n " + str(r["n"]) + ")")
    src_sens.append(r["sensitivity"])
    src_spec.append(r["specificity"])
x = np.arange(len(src_names))
axes[0].bar(x - 0.2, src_sens, width=0.4, color=COLORS["teal"], label="sensitivity")
axes[0].bar(x + 0.2, src_spec, width=0.4, color=COLORS["purple"], label="specificity")
axes[0].set_xticks(x)
axes[0].set_xticklabels(src_names)
axes[0].set_ylim(0, 1.05)
axes[0].set_title("Test performance by source")
axes[0].legend()
grade_names = []
grade_sens = []
for r in grade_rows:
    grade_names.append("Grade " + str(r["grade"]) + "\n(" + str(r["caught"]) + "/" + str(r["total"]) + ")")
    grade_sens.append(r["sensitivity"])
axes[1].bar(np.arange(len(grade_names)), grade_sens, color=COLORS["blue"])
axes[1].set_xticks(np.arange(len(grade_names)))
axes[1].set_xticklabels(grade_names)
axes[1].axhline(TARGET_SENSITIVITY, color=op_color, linestyle="--")
axes[1].set_ylim(0, 1.05)
axes[1].set_title("Test sensitivity by grade (label 1 images)")
save_fig(fig, "05", "source_and_grade", run)

# 06 test metrics with 95% confidence intervals
metric_names = ["sensitivity", "specificity", "precision", "f1", "auroc", "pr_auc"]
metric_values = [t_sens, t_spec, t_prec, t_f1, t_auc, t_prauc]
fig, ax = plt.subplots(figsize=(9, 5))
for i in range(len(metric_names)):
    low = ci[metric_names[i]][0]
    high = ci[metric_names[i]][1]
    ax.errorbar(metric_values[i], i, xerr=[[metric_values[i] - low], [high - metric_values[i]]],
                fmt="o", color=COLORS["blue"], capsize=5)
    ax.text(high + 0.01, i, str(round(metric_values[i], 3)), va="center", fontsize=10)
ax.set_yticks(range(len(metric_names)))
ax.set_yticklabels(metric_names)
ax.invert_yaxis()
ax.set_xlim(0, 1.05)
ax.set_title("Test metrics with 95% confidence intervals (by patient group)")
save_fig(fig, "06", "metrics_ci", run)

# 07 false-negative gallery: the 12 missed DR images with the lowest scores
missed = []
for i in range(len(test_labels)):
    if test_labels[i] == 1 and test_pred[i] == 0:
        missed.append((float(test_probs[i]), i))
missed.sort()
shown = missed[:12]
test_dataset_paths = RetinaDataset(test_df, IMAGE_FOLDER, train=False).paths
fig, axes = plt.subplots(3, 4, figsize=(14, 10.5))
for k in range(12):
    ax = axes[k // 4][k % 4]
    ax.axis("off")
    if k < len(shown):
        score, idx = shown[k]
        ax.imshow(Image.open(test_dataset_paths[idx]).convert("RGB"))
        ax.set_title("grade " + str(test_df["grade"].iloc[idx]) + " | score " + str(round(score, 3)), fontsize=10)
fig.suptitle("Missed DR images (false negatives) with the lowest scores")
save_fig(fig, "07", "false_negative_gallery", run)

print("")
print("Files in", OUT_DIR, ":")
for fname in sorted(os.listdir(OUT_DIR)):
    print("  ", fname)
